# 10 — Marketing Budget Optimization (Team 4)

**Goal:** keep the total monthly advertising budget fixed while re-allocating it across six marketing channels to maximize the *MMM-predicted steady-state incremental revenue*. This notebook reuses Team 2's response functions—there is no need to wait for Team 3 or retrain the MMM.

**Units:** all input/output spend and revenue are **LKR per month**; charts convert figures to millions for readability.

## 1. Imports and paths

You may start Jupyter from the project root or the `notebooks/` folder. The cell below supports both.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.mmm_response import ResponseModel, LABELS
from src.budget_optimization import (build_bounds, optimize_budget,
    allocation_comparison, evaluate_scenarios, sensitivity_analysis, run_pipeline)

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Load the MMM team's response functions and historical spends

The final MMM is calibrated using `models/mmm/final_mmm.json`. `current_spend` is the model's recorded **2025 average monthly spending** across six channels.

In [ ]:
model = ResponseModel(ROOT / "models" / "mmm" / "final_mmm.json")
history = pd.read_csv(ROOT / "data" / "processed" / "mmm_analysis_ready.csv")
current = model.current_spend
fixed_budget_lkr = sum(current.values())
current_incremental_revenue_lkr = model.total_revenue(current)
print(f"Budget: LKR {fixed_budget_lkr:,.2f}")
print(f"Current MMM incremental revenue: LKR {current_incremental_revenue_lkr:,.2f}")
print(f"Current portfolio ROAS: {current_incremental_revenue_lkr/fixed_budget_lkr:.3f}x")

## 3. Specify defensible minimum/maximum channel spend limits

**Assumption to review with the team:** each channel may move by up to 30% relative to its 2025 average, but not outside its recorded historical monthly spend range. The limits prevent unsupported extrapolation and prohibit cutting Radio to zero just because its fitted coefficient is zero.

In [ ]:
bounds = build_bounds(model, history, lower_ratio=0.70, upper_ratio=1.30, cap_to_history=True)
bound_table = pd.DataFrame([
    {"Channel": LABELS[c], "Min LKR M": bounds[c][0]/1e6,
     "Current LKR M": current[c]/1e6, "Max LKR M": bounds[c][1]/1e6}
    for c in model.channels
])
display(bound_table)

## 4. Define the optimisation problem

\[\max_{x_1,\ldots,x_6}\sum_{i=1}^{6} R_i(x_i)\]

Subject to \(\sum_i x_i = B\) and \(L_i \le x_i \le U_i\).

The response \(R_i\) is Team 2's fitted Hill saturation model and uses *steady-state* normalised adstock. We solve it with SciPy SLSQP and its analytic gradient. With the final model's Hill alpha=1 and nonnegative media coefficients, it is a concave revenue objective.

In [ ]:
result = optimize_budget(model, bounds, fixed_budget_lkr)
comparison = allocation_comparison(model, result)
assert np.isclose(sum(result.allocation.values()), fixed_budget_lkr, atol=0.1)
assert result.incremental_revenue_lkr >= current_incremental_revenue_lkr - 0.1
print(result.message)
print(f"Optimized predicted incremental revenue: LKR {result.incremental_revenue_lkr:,.2f}")
print(f"Predicted uplift: LKR {result.incremental_revenue_lkr-current_incremental_revenue_lkr:,.2f} "
      f"({100*(result.incremental_revenue_lkr/current_incremental_revenue_lkr-1):.2f}%)")
print(f"Optimized portfolio ROAS: {result.portfolio_roas:.3f}x")
display(comparison[["channel", "current_spend_lkr", "optimized_spend_lkr", "spend_change_pct",
                    "current_marginal_roas", "optimized_marginal_roas"]])

## 5. Chart: current vs optimal spend

In [ ]:
comparison.set_index("channel")[["current_spend_lkr", "optimized_spend_lkr"]].div(1e6).plot(
    kind="bar", figsize=(11, 5), rot=15,
    title="Current vs optimized monthly spend (LKR million)")
plt.ylabel("Monthly spend (LKR million)")
plt.tight_layout()
plt.show()

## 6. Budget scenarios (±10%, ±20%)

Run the same optimizer with a changed total budget. **These changes mix budget-level effects with reallocation effects**; the fixed-budget +4.41% result is the clean reallocation comparison.

In [ ]:
scenarios, scenario_allocations = evaluate_scenarios(model, bounds)
display(scenarios)
scenarios.plot(x="budget_lkr", y="predicted_incremental_revenue_lkr", marker="o",
               figsize=(8, 4), legend=False, title="Optimized revenue vs available budget")
plt.ylabel("Modelled monthly incremental revenue (LKR)")
plt.xlabel("Total monthly advertising budget (LKR)")
plt.grid(alpha=0.2)
plt.show()

## 7. Sensitivity checks

Do recommendations still look useful when channel spend limits change or when the model is fitted only on the training period? This is an exploratory robustness test—not a confidence interval.

In [ ]:
constraint_sensitivity, model_sensitivity = sensitivity_analysis(model, history)
display(constraint_sensitivity)
display(model_sensitivity)

## 8. Export results for Team 3, the dashboard, and final report

Exports include tidy CSVs and presentation figures. These are generated from the same code as the standalone CLI.

In [ ]:
final_result, summary = run_pipeline(
    output_dir=ROOT / "outputs" / "budget_optimization",
    figures_dir=ROOT / "reports" / "figures")
display(summary)
print("Saved CSV tables under outputs/budget_optimization/ and charts under reports/figures/10_*.png")

## 9. Interpretation and limits

- Maximize *MMM-estimated incremental revenue*, not profit. Profit ROI requires valid gross/contribution margins and other costs.
- This scenario represents monthly *steady-state* spending (normalised adstock), not the first month following changes.
- Radio's coefficient is zero in this fit, which should not be interpreted as evidence that Radio truly has no effect.
- Monthly MMM data include only 36 rows, with potential media collinearity and attribution uncertainty. Treat results as a prioritised **pilot hypothesis**, not a guaranteed improvement.
- Validate feasibility of minimum/maximum channel spends with the business and monitor actual experimental outcomes before rollout.